# CAFE — Generalizable Facial Expression Recognition
### Korea University Team 6 | Based on ECCV 2024 — Zhang et al.

**Improvement:** AU-guided channel relevance weighting on l_sep via 512-channel cosine similarity  
**Landmark model:** face_alignment (GPU batch inference)  
**Pipeline:** Input → CLIP (frozen) + ResNet-18 (trained) → masked_features → AU-weighted l_sep + l_div → loss

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade numba
!pip install ftfy regex tqdm kagglehub==0.3.6 pandas scikit-learn matplotlib
!pip install git+https://github.com/openai/CLIP.git
!pip install face-alignment && pip uninstall opencv-python -y && pip install --force-reinstall --no-cache-dir opencv-python-headless
!pip install scipy

## 2. Mount Google Drive
Upload `resnet18_msceleb.pth` to a `deeplearning` folder in Drive.  
**Skip this cell if using Elice.**

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#import os
#DRIVE_FOLDER = '/content/drive/MyDrive/deeplearning'

## 3. File Paths
Set paths to model weights and dataset.

In [ ]:
import os
import kagglehub

if os.path.exists('/content/drive'):
  FILE_FOLDER = DRIVE_FOLDER
else:
  FILE_FOLDER = '.'

# ── ResNet-18 MS-Celeb weights (from Drive or Local) ──────────────────────────
MSCELEB_PATH = os.path.join(FILE_FOLDER, 'resnet18_msceleb.pth')

# ── Copy kaggle.json from Drive ──────────────────────────────────────
!mkdir -p ~/.kaggle
!cp "{FILE_FOLDER}/kaggle.json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# ── Download dataset from Kaggle ─────────────────────────────────────

# RAF-DB
#train_dataset_path = kagglehub.dataset_download("shuvoalok/raf-db-dataset")

# FERPlus ___________________________________________________________________________________ UNCOMMENT for FERplus
#train_dataset_path = kagglehub.dataset_download("arnabkumarroy02/ferplus")

# affectnet
# train_dataset_path = kagglehub.dataset_download("mstjebashazida/affectnet")

# MMA
#train_dataset_path = kagglehub.dataset_download("mahmoudima/mma-facial-expression")

# SFEW
train_dataset_path = kagglehub.dataset_download("vlntnstarodub/datasetsfew")

# ────────────────────────────────────────────────────────────────────

print('MS-Celeb weights found :', os.path.exists(MSCELEB_PATH))
print('Train Dataset path           :', train_dataset_path)

# See what's inside
print('\n Train Dataset contents:')
for f in os.listdir(train_dataset_path):
    print(' ', f)

Set test dataset path.

In [ ]:
# ── Download dataset from Kaggle ─────────────────────────────────────

# RAF-DB
test_dataset_path = kagglehub.dataset_download("shuvoalok/raf-db-dataset")

# FERPlus ___________________________________________________________________________________ UNCOMMENT for FERplus
#test_dataset_path = kagglehub.dataset_download("arnabkumarroy02/ferplus")

# affectnet
#test_dataset_path = kagglehub.dataset_download("mstjebashazida/affectnet")

# MMA
#test_dataset_path = kagglehub.dataset_download("mahmoudima/mma-facial-expression")

# SFEW
#test_dataset_path = kagglehub.dataset_download("vlntnstarodub/datasetsfew")

# ────────────────────────────────────────────────────────────────────

print('MS-Celeb weights found :', os.path.exists(MSCELEB_PATH))
print('Test Dataset path           :', test_dataset_path)

# See what's inside
print('\n Test Dataset contents:')
for f in os.listdir(test_dataset_path):
    print(' ', f)

## 4. Imports & All Dependencies

In [ ]:
import os
import cv2
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision import transforms
from torch.autograd import Variable
from torch.nn.modules.module import Module
from torch.nn.modules.utils import _pair
from torch.nn.parameter import Parameter
from torch.utils.data import random_split
from collections import Counter

import face_alignment
from scipy.ndimage import gaussian_filter
import clip
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load CLIP model (frozen)
clip_model, preprocess = clip.load('ViT-B/32', device=device)
print('CLIP loaded.')

# Load face_alignment landmark detector (GPU)
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, device=str(device))
print('face_alignment loaded.')

## 4a. Label Maps
Unified label mapping across datasets (0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral).

In [ ]:
raf_map = {
    '1': 5,  # surprise
    '2': 2,  # fear
    '3': 1,  # disgust
    '4': 3,  # happy
    '5': 4,  # sad
    '6': 0,  # angry
    '7': 6   # neutral
}

ferplus_map = {
    'fear': 2,
    'suprise': 5,
    'surprise': 5,  # test folder correct spelling
    'angry': 0,
    'neutral': 6,
    'sad': 4,
    'disgust': 1,
    'happy': 3
}

affectnet_train_map = {
    'anger': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

affectnet_test_map = {
    'Anger': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

sfew_map = {
    'Angry': 0,
    'Disgust': 1,
    'Fear': 2,
    'Happy': 3,
    'Neutral': 6,
    'Sad': 4,
    'Surprise': 5
}

mma_map = {
    'angry': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

## 5. Dataset Loader
Uncomment the dataset you want to train on.

In [ ]:
# ── DATASET PATHS — switch by commenting/uncommenting ───────────────

# RAF-DB #___________________________________________________________________________________ UNCOMMENT for RAF-DB
#TRAIN_DATASET_PATH = os.path.join(train_dataset_path, 'DATASET')
#TRAIN_DATASET_NAME = 'RAFDB'
#TRAIN_MAP = raf_map
#TRAIN_PHASE = 'train'

# FERPlus #___________________________________________________________________________________ UNCOMMENT for FERplus
#TRAIN_DATASET_PATH = train_dataset_path
#TRAIN_DATASET_NAME = 'FERPlus'
#TRAIN_MAP = ferplus_map
#TRAIN_PHASE = 'train'
#VALIDATION_PHASE = 'validation'

#AffectNET
# TRAIN_DATASET_PATH = os.path.join(train_dataset_path,'archive (3)')
# TRAIN_DATASET_NAME = 'AffectNET'
# TRAIN_MAP = affectnet_train_map
# TRAIN_PHASE = 'Train'

# MMA
#TRAIN_DATASET_PATH = os.path.join(train_dataset_path,'MMAFEDB')
#TRAIN_DATASET_NAME = 'MMA'
# TRAIN_MAP = mma_map
# TRAIN_PHASE = 'train'
# VALIDATION_PHASE = 'valid'

# SFEW
TRAIN_DATASET_PATH = train_dataset_path
TRAIN_DATASET_NAME = 'SFEW'
TRAIN_MAP = sfew_map
TRAIN_PHASE = 'Train'

# ────────────────────────────────────────────────────────────────────

# Class names — same for both datasets (7 basic expressions)
# RAF-DB label order (1-indexed in txt, we subtract 1):
# 1=Surprise, 2=Fear, 3=Disgust, 4=Happy, 5=Sad, 6=Angry, 7=Neutral
CLASS_NAMES = ['surprise', 'fear', 'disgust', 'happy', 'sad', 'angry', 'neutral']
NUM_CLASSES = 7

print(f'Using dataset: {TRAIN_DATASET_NAME}')
print(f'Path: {TRAIN_DATASET_PATH}')

Set test dataset.

In [ ]:
# ── DATASET PATHS — switch by commenting/uncommenting ───────────────

# RAF-DB #___________________________________________________________________________________ UNCOMMENT for RAF-DB
TEST_DATASET_PATH = os.path.join(test_dataset_path, 'DATASET')
TEST_DATASET_NAME = 'RAFDB'
TEST_MAP = raf_map
TEST_PHASE = 'test'

# FERPlus #___________________________________________________________________________________ UNCOMMENT for FERplus
#TEST_DATASET_PATH = test_dataset_path
#TEST_DATASET_NAME = 'FERPlus'
#TEST_MAP = ferplus_map
#TEST_PHASE = 'test'

#AffectNET
#TEST_DATASET_PATH = os.path.join(test_dataset_path,'archive (3)')
#TEST_DATASET_NAME = 'AffectNET'
#TEST_MAP = affectnet_test_map
#TEST_PHASE = 'Test'

# MMA
#TEST_DATASET_PATH = os.path.join(test_dataset_path,'MMAFEDB')
#TEST_DATASET_NAME = 'MMA'
#TEST_MAP = mma_map
#TEST_PHASE = 'test'

# SFEW
#TEST_DATASET_PATH = test_dataset_path
#TEST_DATASET_NAME = 'SFEW'
#TEST_MAP = sfew_map
#TEST_PHASE = 'Val'

# ────────────────────────────────────────────────────────────────────

# Class names — same for both datasets (7 basic expressions)
# RAF-DB label order (1-indexed in txt, we subtract 1):
# 1=Surprise, 2=Fear, 3=Disgust, 4=Happy, 5=Sad, 6=Angry, 7=Neutral
CLASS_NAMES = ['surprise', 'fear', 'disgust', 'happy', 'sad', 'angry', 'neutral']
NUM_CLASSES = 7

print(f'Using dataset: {TEST_DATASET_NAME}')
print(f'Path: {TEST_DATASET_PATH}')

## 5a. Augmentation & Transforms

In [ ]:
def add_g(image_array, mean=0.0, var=30):
    """Add Gaussian noise"""
    std = var ** 0.5
    image_add = image_array + np.random.normal(mean, std, image_array.shape)
    image_add = np.clip(image_add, 0, 255).astype(np.uint8)
    return image_add

def flip_image(image_array):
    return cv2.flip(image_array, 1)

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


class FolderDataset(data.Dataset):

    def __init__(self, root, phase='train', transform=None, label_map=None):
        self.transform = transform
        self.aug_func  = [flip_image, add_g]
        self.phase     = phase
        self.label_map = label_map

        split_dir = os.path.join(root, phase)
        self.classes = sorted([c for c in os.listdir(split_dir) if (c != 'contempt' and c != 'Contempt')])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.file_paths = []
        self.labels     = []

        for cls in self.classes:
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.file_paths.append(os.path.join(cls_dir, fname))
                    if self.label_map is not None:
                      mapped_label = self.label_map[cls]
                    else:
                      mapped_label = self.class_to_idx[cls]

                    self.labels.append(mapped_label)

        print(f'[{phase}] {len(self.file_paths)} samples | classes: {self.classes}')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        label = self.labels[idx]
        image = cv2.imread(self.file_paths[idx])
        image = image[:, :, ::-1]  # BGR → RGB

        # Augmentation during training
        if self.phase.lower() == 'train':
            if random.uniform(0, 1) > 0.5:
                image = add_g(image)

        if self.transform is not None:
            image = self.transform(image)

        # Flipped image (used by some loss variants)
        image_flip = transforms.RandomHorizontalFlip(p=1)(image)

        return image, label, idx, image_flip

In [ ]:
from torch.utils.data import random_split

# Transforms — from original CAFE code
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomErasing(scale=(0.02, 0.25))
])

eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Dataset and Loaders ──────────────────────────────────────────────
BATCH_SIZE = 32
NUM_WORKERS = 8

full_train_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=TRAIN_PHASE, transform=train_transforms, label_map=TRAIN_MAP)
test_dataset  = FolderDataset(TEST_DATASET_PATH, phase=TEST_PHASE,  transform=eval_transforms, label_map=TEST_MAP)

if TRAIN_DATASET_NAME in ['RAFDB', 'AffectNET', 'SFEW']:
    train_size = int(0.8 * len(full_train_dataset))
    valid_size = len(full_train_dataset) - train_size
    g = torch.Generator().manual_seed(42)
    train_dataset, _ = random_split(full_train_dataset, [train_size, valid_size], generator=g)

    # Validation set uses eval_transforms (no augmentation)
    full_eval_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=TRAIN_PHASE,
                                      transform=eval_transforms, label_map=TRAIN_MAP)
    full_eval_dataset.phase = 'eval'
    g = torch.Generator().manual_seed(42)
    _, val_dataset = random_split(full_eval_dataset, [train_size, valid_size], generator=g)
else:
    train_dataset = full_train_dataset
    val_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=VALIDATION_PHASE, transform=eval_transforms, label_map=TRAIN_MAP)

train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True,  num_workers=NUM_WORKERS, pin_memory=False)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader  = data.DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

## 5b. AU-Guided Channel Relevance — 512-Channel Cosine Similarity

For each image: build a 7x7 AU heatmap from face_alignment landmarks + FACS codebook.  
Compare each of the 512 channel maps (7x7) against the heatmap via cosine similarity.  
High-scoring channels receive stronger separation pressure in l_sep.

In [ ]:
# ── AU Codebook + Landmark Heatmap Builder ───────────────────────────────────
# FACS-based mapping: expression -> active AUs -> landmark indices -> 7x7 heatmap

# Expression -> active AUs (FACS codebook)
AU_CODEBOOK = {
    'angry':    [4, 5, 7, 23, 24],
    'disgust':  [9, 15, 16, 25, 26],
    'fear':     [1, 2, 4, 5, 7, 20, 26],
    'happy':    [6, 12],
    'neutral':  [],
    'sad':      [1, 4, 15, 17],
    'surprise': [1, 2, 5, 26, 27],
    'suprise':  [1, 2, 5, 26, 27],  # FERPlus typo
}

# AU -> 68-point landmark indices (compatible with face_alignment)
AU_TO_DLIB = {
    1:  [17, 18, 19, 21, 22, 23, 24, 26],  # inner brow
    2:  [17, 18, 19, 20, 23, 24, 25, 26],  # outer brow
    4:  [19, 20, 21, 22, 23, 24],           # brow lowerer
    5:  [36, 37, 38, 43, 44, 45],           # upper lid (eyes)
    6:  [1, 2, 14, 15],                     # cheek raiser
    7:  [36, 39, 42, 45],                   # lid tightener
    9:  [27, 28, 29, 30],                   # nose wrinkler
    12: [48, 54, 49, 53],                   # lip corner puller (smile)
    15: [48, 54, 56, 58],                   # lip corner depressor
    16: [56, 57, 58],                       # lower lip depressor
    17: [6, 7, 8, 57],                      # chin raiser
    20: [48, 54],                           # lip stretcher
    23: [60, 61, 62, 63, 64],              # lip tightener
    24: [60, 61, 62, 63, 64],              # lip pressor
    25: [60, 61, 62, 63, 64, 65, 66, 67], # lips part
    26: [6, 7, 8, 56, 57, 58],            # jaw drop
    27: [6, 7, 8, 56, 57, 58],            # mouth stretch
}

SYNONYM_MAP = {'anger': 'angry', 'happiness': 'happy', 'sadness': 'sad', 'surprised': 'surprise'}

GRID_SIZE = 7
IMG_SIZE  = 224
SCALE     = GRID_SIZE / IMG_SIZE

# Unified label index -> expression name
# 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
LABEL_TO_CLASS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

print('AU codebook defined.')
print(f'Grid scale: {IMG_SIZE}px image -> {GRID_SIZE}x{GRID_SIZE} feature map')

In [ ]:
# ── Per-Image AU Heatmap Builder (face_alignment) ────────────────────────────
# Builds a smooth 7x7 AU heatmap per image using:
#   1. face_alignment -> 68 landmark (x,y) coordinates
#   2. AU codebook -> which landmarks are active for this expression
#   3. Gaussian smoothing -> soft 7x7 heatmap
# Output shape: (7, 7) tensor, values in [0, 1]

_MEAN = np.array([0.485, 0.456, 0.406])
_STD  = np.array([0.229, 0.224, 0.225])

# ── Landmark detection statistics ────────────────────────────────────
landmark_stats = {'total': 0, 'success': 0, 'fail': 0, 'neutral_skip': 0}

def reset_landmark_stats():
    landmark_stats['total'] = 0
    landmark_stats['success'] = 0
    landmark_stats['fail'] = 0
    landmark_stats['neutral_skip'] = 0

def denormalize_to_uint8(img_tensor):
    """(C, H, W) normalized tensor -> (H, W, 3) uint8 RGB numpy array"""
    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = (img_np * _STD + _MEAN) * 255.0
    return np.clip(img_np, 0, 255).astype(np.uint8)

def build_heatmap_from_landmarks(img_np, expression_name):
    """
    img_np          : (224, 224, 3) uint8 RGB
    expression_name : string e.g. 'happy'
    Returns         : (7, 7) float32 tensor, Gaussian-smoothed, normalized to [0,1]
                      Returns uniform map if no face detected or neutral expression.
    """
    landmark_stats['total'] += 1

    expr = SYNONYM_MAP.get(expression_name, expression_name)
    active_aus = AU_CODEBOOK.get(expr, [])

    # neutral -> uniform heatmap
    if not active_aus:
        landmark_stats['neutral_skip'] += 1
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)

    # face_alignment landmark extraction
    try:
        landmarks_list = fa.get_landmarks(img_np)
    except Exception:
        landmark_stats['fail'] += 1
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)

    if landmarks_list is None or len(landmarks_list) == 0:
        landmark_stats['fail'] += 1
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)

    landmark_stats['success'] += 1
    landmarks = landmarks_list[0]  # (68, 2)
    img_h, img_w = img_np.shape[:2]

    # Build heatmap from active AU landmark indices
    heatmap = np.zeros((GRID_SIZE, GRID_SIZE), dtype=np.float32)
    for au in active_aus:
        lm_indices = AU_TO_DLIB.get(au, [])
        for li in lm_indices:
            if li < len(landmarks):
                lx, ly = landmarks[li]
                gc = int(np.clip(lx * GRID_SIZE / img_w, 0, GRID_SIZE - 1))
                gr = int(np.clip(ly * GRID_SIZE / img_h, 0, GRID_SIZE - 1))
                heatmap[gr, gc] += 1.0

    heatmap = gaussian_filter(heatmap, sigma=0.8)
    hmax = heatmap.max()
    if hmax > 0:
        heatmap = heatmap / hmax

    return torch.from_numpy(heatmap)

print('Per-image AU heatmap builder defined (face_alignment + FACS codebook).')
print('Landmark failure tracking enabled.')

In [ ]:
# ── 512-Channel Cosine Similarity Relevance Scorer ───────────────────────────
# Core contribution: scores each of 512 channels by how much its 7×7
# activation map resembles the AU heatmap — no grid cell collapse.
#
# spatial_map : (N, 512, 7, 7) — ResNet Layer4 output
# au_heatmaps : (N, 7, 7)      — per-image AU heatmaps from landmarks
# Returns     : (N, 512)        — softmax-normalized relevance weights

def compute_channel_relevance_512(spatial_map, au_heatmaps):
    N, C, H, W = spatial_map.shape   # N, 512, 7, 7

    # Flatten spatial dims for cosine similarity: (N, 512, 49)
    feat_flat = spatial_map.view(N, C, -1)          # (N, 512, 49)

    # AU heatmap: (N, 7, 7) → (N, 1, 49) → broadcast to (N, 512, 49)
    au_flat   = au_heatmaps.view(N, 1, -1).expand(N, C, H*W)  # (N, 512, 49)

    # Cosine similarity per channel: dot / (norm_feat * norm_au)
    dot       = (feat_flat * au_flat).sum(dim=2)                      # (N, 512)
    norm_feat = feat_flat.norm(dim=2).clamp(min=1e-8)                 # (N, 512)
    norm_au   = au_flat.norm(dim=2).clamp(min=1e-8)                   # (N, 512)
    scores    = dot / (norm_feat * norm_au)                           # (N, 512)

    # Softmax: convert scores to attention weights summing to 1
    relevance_weights = F.softmax(scores, dim=1)                      # (N, 512)
    return relevance_weights

print('512-channel cosine similarity relevance scorer defined.')

In [ ]:
# ── AU-Weighted supervisor() ─────────────────────────────────────────────────
# l_sep weighted by AU cosine similarity scores | l_div unchanged

def supervisor_au_weighted(x, targets, relevance_weights, cnum=73):
    # ── l_div — unchanged from original CAFE ─────────────────────────
    branch   = x.reshape(x.size(0), x.size(1), 1, 1)
    branch   = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch)
    branch   = branch.reshape(x.size(0), -1, 1)
    loss_div = 1.0 - torch.mean(torch.sum(branch, dim=1)) / cnum

    # ── l_sep — reweighted by AU channel relevance ───────────────────
    branch_1 = x.reshape(x.size(0), x.size(1), 1, 1)
    branch_1 = branch_1 * relevance_weights.unsqueeze(-1).unsqueeze(-1)
    branch_1 = branch_1 * Mask(x.size(0))
    branch_1 = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch_1)
    branch_1 = branch_1.view(branch_1.size(0), -1)
    loss_sep = nn.CrossEntropyLoss()(branch_1, targets)

    return [loss_sep, loss_div]

print('supervisor_au_weighted defined.')


In [ ]:
# ── Batch AU Heatmap Builder ──────────────────────────────────────────────────
# Builds per-image AU heatmaps for a full batch (face_alignment)

def get_batch_au_heatmaps(imgs_batch, labels_batch):
    heatmaps = []
    for i in range(imgs_batch.size(0)):
        img_np    = denormalize_to_uint8(imgs_batch[i])
        expr_name = LABEL_TO_CLASS[labels_batch[i].item()]
        hm        = build_heatmap_from_landmarks(img_np, expr_name)
        heatmaps.append(hm)
    return torch.stack(heatmaps).to(imgs_batch.device)

print('Batch AU heatmap builder defined (face_alignment).')

## 6. CAFE Model Architecture

In [ ]:
# ── Custom MaxPool2d (used in channel separation) ────────────────────
class my_MaxPool2d(Module):
    def __init__(self, kernel_size, stride=None, padding=0, dilation=1,
                 return_indices=False, ceil_mode=False):
        super(my_MaxPool2d, self).__init__()
        self.kernel_size    = kernel_size
        self.stride         = stride or kernel_size
        self.padding        = padding
        self.dilation       = dilation
        self.return_indices = return_indices
        self.ceil_mode      = ceil_mode

    def forward(self, input):
        input = input.transpose(3, 1)
        input = F.max_pool2d(input, self.kernel_size, self.stride,
                             self.padding, self.dilation, self.ceil_mode,
                             self.return_indices)
        input = input.transpose(3, 1).contiguous()
        return input


# ── Channel dropping mask ────────────────────────────────────────────
def Mask(nb_batch):
    """
    Generates random channel drop mask.
    7 expression groups x (63 or 64 channels each) = 512 total
    Drops 10 channels per group randomly.
    """
    bar = []
    for i in range(7):
        foo = [1] * 63 + [0] * 10
        if i == 6:
            foo = [1] * 64 + [0] * 10  # last group gets 1 extra channel
        random.shuffle(foo)
        bar += foo
    bar = [bar for _ in range(nb_batch)]
    bar = np.array(bar).astype('float32')
    bar = bar.reshape(nb_batch, 512, 1, 1)
    bar = torch.from_numpy(bar).to(device)
    bar = Variable(bar)
    return bar


# ── Channel separation + channel diverse loss ────────────────────────
def supervisor(x, targets, cnum=73):
    """
    Computes l_sep (channel separation loss) and l_div (channel diverse loss).
    x       : masked features (N, 512)
    targets : expression labels (N,)
    cnum    : channels per expression group (73 for groups 0-5, 74 for group 6)
    Returns: [l_sep, l_div]
    """
    # l_div — channel diverse loss
    branch = x
    branch = branch.reshape(branch.size(0), branch.size(1), 1, 1)
    branch = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch)
    branch = branch.reshape(branch.size(0), branch.size(1),
                            branch.size(2) * branch.size(3))
    loss_2 = 1.0 - 1.0 * torch.mean(torch.sum(branch, 2)) / cnum

    # l_sep — channel separation cross-entropy loss
    mask    = Mask(x.size(0))
    branch_1 = x.reshape(x.size(0), x.size(1), 1, 1) * mask
    branch_1 = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch_1)
    branch_1 = branch_1.view(branch_1.size(0), -1)
    loss_1   = nn.CrossEntropyLoss()(branch_1, targets)

    return [loss_1, loss_2]


# ── Custom ResNet-18 BasicBlock ──────────────────────────────────────
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        if downsample:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.downsample = None

    def forward(self, x):
        i = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            i = self.downsample(i)
        x += i
        return self.relu(x)


# ── Custom ResNet ────────────────────────────────────────────────────
class ResNet(nn.Module):
    def __init__(self, block, n_blocks, channels, output_dim):
        super().__init__()
        self.in_channels = channels[0]
        assert len(n_blocks) == len(channels) == 4

        self.conv1   = nn.Conv2d(3, self.in_channels, kernel_size=7,
                                  stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(self.in_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1  = self.get_resnet_layer(block, n_blocks[0], channels[0])
        self.layer2  = self.get_resnet_layer(block, n_blocks[1], channels[1], stride=2)
        self.layer3  = self.get_resnet_layer(block, n_blocks[2], channels[2], stride=2)
        self.layer4  = self.get_resnet_layer(block, n_blocks[3], channels[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(self.in_channels, output_dim)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []
        downsample = (self.in_channels != block.expansion * channels)
        layers.append(block(self.in_channels, channels, stride, downsample))
        for _ in range(1, n_blocks):
            layers.append(block(block.expansion * channels, channels))
        self.in_channels = block.expansion * channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        x = self.fc(h)
        return x, h

print('Utility classes defined (MaxPool2d, Mask, supervisor, BasicBlock, ResNet).')

In [ ]:
# ── CAFE Model — exposes spatial_map and masked_features ─────────────────────

class Model(nn.Module):
    def __init__(self, msceleb_path, num_classes=7, drop_rate=0):
        super(Model, self).__init__()

        res18 = ResNet(block=BasicBlock, n_blocks=[2, 2, 2, 2],
                       channels=[64, 128, 256, 512], output_dim=1000)
        msceleb_model = torch.load(msceleb_path, map_location='cpu', weights_only=False)
        state_dict    = msceleb_model['state_dict']
        res18.load_state_dict(state_dict, strict=False)
        print('MS-Celeb weights loaded.')

        self.drop_rate = drop_rate
        self.features  = nn.Sequential(*list(res18.children())[:-2])    # (N, 512, 7, 7)
        self.features2 = nn.Sequential(*list(res18.children())[-2:-1])  # avgpool
        fc_in_dim      = list(res18.children())[-1].in_features          # 512
        self.fc        = nn.Linear(fc_in_dim, num_classes)

        self.parm = {}
        for name, parameters in self.fc.named_parameters():
            self.parm[name] = parameters

    def forward(self, x, clip_model, targets, phase='train'):
        with torch.no_grad():
            image_features = clip_model.encode_image(x).float()  # (N, 512)

        spatial_map     = self.features(x)                        # (N, 512, 7, 7)
        x_pool          = self.features2(spatial_map)             # (N, 512, 1, 1)
        x_flat          = x_pool.view(x_pool.size(0), -1)        # (N, 512)
        masked_features = image_features * torch.sigmoid(x_flat) # (N, 512)
        out             = self.fc(masked_features)                # (N, 7)

        if phase == 'train':
            return out, spatial_map, masked_features
        else:
            return out, out

print('Model defined (returns spatial_map + masked_features in train mode).')

## 7. Initialize Model, Optimizer, Scheduler

In [ ]:
setup_seed(3407)

model = Model(msceleb_path=MSCELEB_PATH, num_classes=NUM_CLASSES)
model.to(device)

# From original CAFE code
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

EPOCHS = 20

print(f'\nOptimizer : Adam  lr=0.0002, wd=1e-4')
print(f'Scheduler : ExponentialLR gamma=0.9')
print(f'Epochs    : {EPOCHS}')

## 8. Train and Test Functions

In [ ]:
from collections import Counter
label_counts = Counter(full_train_dataset.labels)
counts = torch.tensor([label_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)
class_weights = counts.sum() / (NUM_CLASSES * counts)
class_weights = torch.clamp(class_weights, max=3.0).to(device)

In [ ]:
# ── Train and Test Functions ──────────────────────────────────────────────────
# Mode 1: Original CAFE | Mode 2: Class Imbalance only | Mode 3: AU-weighted l_sep (active)

def train_one_epoch(model, train_loader, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct_sum  = 0

    reset_landmark_stats()

    for imgs, labels, _, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # ── MODE 1: Original CAFE ────────────────────────────────────────────
        #output, spatial_map, masked_features = model(imgs, clip_model, labels, phase='train')
        #mc_loss  = supervisor(masked_features, labels, cnum=73)
        #loss_cls = nn.CrossEntropyLoss()(output, labels)
        #loss = loss_cls + 1.5 * mc_loss[0] + 5.0 * mc_loss[1]

        # ── MODE 2: Class Imbalance Loss only ────────────────────────────────
        #output, spatial_map, masked_features = model(imgs, clip_model, labels, phase='train')
        #mc_loss  = supervisor(masked_features, labels, cnum=73)
        #loss_cls = nn.CrossEntropyLoss(weight=class_weights)(output, labels)
        #loss = loss_cls + 1.5 * mc_loss[0] + 5.0 * mc_loss[1]

        # ── MODE 3: AU-weighted l_sep (active) ───────────────────────────────
        output, spatial_map, masked_features = model(imgs, clip_model, labels, phase='train')
        au_heatmaps = get_batch_au_heatmaps(imgs, labels)
        with torch.no_grad():
            relevance_weights = compute_channel_relevance_512(spatial_map.detach(), au_heatmaps)
        mc_loss  = supervisor_au_weighted(masked_features, labels, relevance_weights, cnum=73)
        loss_cls = nn.CrossEntropyLoss()(output, labels)
        loss     = loss_cls + 1.5 * mc_loss[0] + 5.0 * mc_loss[1]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicts  = torch.max(output, 1)
        correct_sum += torch.eq(predicts, labels).sum()
        running_loss += loss.item()

    scheduler.step()

    # Compute landmark fail rate (excluding neutral)
    f = landmark_stats['fail']
    n = landmark_stats['neutral_skip']
    non_neutral = landmark_stats['total'] - n
    fail_rate = f / max(non_neutral, 1) * 100

    acc = (correct_sum.float() / len(train_loader.dataset)).item()
    loss_avg = running_loss / len(train_loader)
    return acc, loss_avg, fail_rate


def test_model(model, test_loader, device):
    model.eval()
    with torch.no_grad():
        running_loss = 0.0
        iter_cnt     = 0
        correct_sum  = 0
        data_num     = 0

        for imgs, labels, indexes, imgs_flip in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs, _ = model(imgs, clip_model, labels, phase='test')
            loss        = nn.CrossEntropyLoss()(outputs, labels)
            iter_cnt    += 1
            _, predicts  = torch.max(outputs, 1)
            correct_sum += torch.eq(predicts, labels).sum()
            running_loss += loss
            data_num     += outputs.size(0)

        running_loss = running_loss / iter_cnt
        test_acc     = correct_sum.float() / float(data_num)

    return test_acc.item(), running_loss.item()

print('Train and test functions ready. Active: MODE 3')

In [ ]:
# Check GPU
!nvidia-smi

# Check batch size and how many batches per epoch
print(f'Batch size     : {BATCH_SIZE}')
print(f'Train batches  : {len(train_loader)}')
print(f'Estimated time : {len(train_loader) * 3} seconds per epoch')

## 9. Training Loop
Only run when training. Skip if only test dataset changed.

In [ ]:
if os.path.exists('/content/drive'):
    SAVE_DIR = '/content/drive/MyDrive/deeplearning'
else:
    SAVE_DIR = './'

history  = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
best_acc = 0.0
all_fail_rates = []

print('=' * 60)
print(f'Training CAFE on {TRAIN_DATASET_NAME}')
print(f'Loss: l_cls + 1.5 * l_sep_AU_weighted + 5.0 * l_div')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    train_acc, train_loss, fail_rate = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_acc,   val_loss   = test_model(model, val_loader, device)
    all_fail_rates.append(fail_rate)

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)

    print(f'Epoch [{epoch:>3}/{EPOCHS}] '
          f'Train Acc: {train_acc*100:5.2f}% Loss: {train_loss:.4f} | '
          f'Val Acc: {val_acc*100:5.2f}% Loss: {val_loss:.4f} | '
          f'Landmark Fail: {fail_rate:.1f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'model_state_dict': model.state_dict()},
                    f'ours_best_{TRAIN_DATASET_NAME}.pth')
        torch.save({'model_state_dict': model.state_dict()},
                    os.path.join(SAVE_DIR, f'ours_best_{TRAIN_DATASET_NAME}.pth'))
        print(f'  -> Best model saved (acc={best_acc*100:.2f}%)')

    torch.save({'model_state_dict': model.state_dict()},
                f'ours_final_{TRAIN_DATASET_NAME}.pth')
    torch.save({'model_state_dict': model.state_dict()},
                os.path.join(SAVE_DIR, f'ours_final_{TRAIN_DATASET_NAME}.pth'))

    with open('results.txt', 'a') as f:
        f.write(f'{epoch}_{val_acc:.4f}\n')

avg_fail = sum(all_fail_rates) / len(all_fail_rates)
print('=' * 60)
print(f'Training complete. Best val acc: {best_acc*100:.2f}%')
print(f'Average landmark fail rate: {avg_fail:.1f}%')

In [ ]:
import matplotlib.pyplot as plt

epochs_x = list(range(1, len(history['train_acc']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epochs_x, [a * 100 for a in history['train_acc']], 'b-o',
             markersize=3, label='Train')
axes[0].plot(epochs_x, [a * 100 for a in history['val_acc']],  'r-o',
             markersize=3, label='Val')
axes[0].set_title(f'Accuracy — {TRAIN_DATASET_NAME}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(epochs_x, history['train_loss'], 'b-o', markersize=3, label='Train')
axes[1].plot(epochs_x, history['val_loss'],  'r-o', markersize=3, label='Val')
axes[1].set_title(f'Loss — {TRAIN_DATASET_NAME}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('cafe_training_curves.png', dpi=150)
plt.show()
print(f'Best val accuracy: {best_acc * 100:.2f}%')

## 10. Load Best Model and Run Final Evaluation

In [ ]:
if os.path.exists('/content/drive'):
  SAVE_DIR = '/content/drive/MyDrive/deeplearning'
else:
  SAVE_DIR = './'

print('Evaluating on test set', TEST_DATASET_NAME)

# load best model
model_path = os.path.join(SAVE_DIR, f'ours_best_{TRAIN_DATASET_NAME}.pth')
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_acc, test_loss = test_model(model, test_loader, device)
print(f'Loading model: ours_best_{TRAIN_DATASET_NAME}.pth')
print(f'Testing on  : {TEST_DATASET_NAME}')

print(f'Final Test Acc: {test_acc*100:.2f}% Loss: {test_loss:.4f}')

In [ ]:
test_acc, test_loss = test_model(model, test_loader, device)
print(f'Testing on  : {TEST_DATASET_NAME}')
print(f'Test Acc: {test_acc*100:.2f}% Loss: {test_loss:.4f}')

## 11. Per-Class Accuracy & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels, _, image_flip in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Pass clip_model and labels as required by Model.forward
        # The image_flip tensor is not used in the model's forward pass during evaluation
        outputs, _ = model(images, clip_model, labels, phase='test')
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Classification Report
class_names = [                 
    'Angry',                   # 0
    'Disgust',                 # 1
    'Fear',                    # 2
    'Happy',                   # 3
    'Sad',                     # 4
    'Surprise',                # 5 
    'Neutral'                  # 6
]


In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
import matplotlib.pylab as plt

print(classification_report(all_labels, all_preds, target_names=class_names))
matrix = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(all_labels, all_preds), display_labels=class_names)

matrix.plot(cmap = 'Blues', xticks_rotation = 90)
plt.title(f"Confusion Matrix on Test Dataset {TEST_DATASET_NAME}")
plt.show()

## 11a. AU Heatmap & Channel Relevance Visualization

Verify that AU-weighted l_sep is working correctly:
1. **AU heatmap** — Does the landmark-based AU heatmap point to different regions per expression?
2. **Channel relevance distribution** — Do AU-related channels actually receive higher scores among 512 channels?
3. **Per-expression relevance comparison** — Are important channels different across expressions?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

model.eval()

# --- 1. Per-expression AU heatmap visualization ---
class_names_display = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Find one sample per expression class
samples_by_class = {}
with torch.no_grad():
    for imgs, labels, idxs, _ in test_loader:
        for i in range(imgs.size(0)):
            label = labels[i].item()
            if label not in samples_by_class:
                samples_by_class[label] = (imgs[i:i+1], labels[i:i+1])
        if len(samples_by_class) >= 7:
            break

available_classes = sorted(samples_by_class.keys())
n_classes = len(available_classes)
print(f'Found {n_classes} classes in test set: {[class_names_display[c] for c in available_classes]}')

fig, axes = plt.subplots(1, n_classes, figsize=(3 * n_classes, 3))
if n_classes == 1:
    axes = [axes]

for plot_idx, cls_idx in enumerate(available_classes):
    img, label = samples_by_class[cls_idx]
    img_dev = img.to(device)
    label_dev = label.to(device)
    au_hm = get_batch_au_heatmaps(img_dev, label_dev)
    axes[plot_idx].imshow(au_hm[0].cpu().numpy(), cmap='hot', interpolation='nearest')
    axes[plot_idx].set_title(class_names_display[cls_idx], fontsize=12)
    axes[plot_idx].axis('off')

fig.suptitle('Expression-specific AU Heatmaps (7x7)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('-> Check if AU heatmaps point to different spatial regions per expression')

In [ ]:
# --- 2. Channel Relevance Distribution: uniform vs differentiated ---
model.eval()

all_relevance = {i: [] for i in range(7)}
MAX_BATCHES = 30  # limit batches for speed (face_alignment is slow per-image)

with torch.no_grad():
    for batch_idx, (imgs, labels, idxs, _) in enumerate(tqdm(test_loader, desc='Computing relevance', total=min(MAX_BATCHES, len(test_loader)))):
        if batch_idx >= MAX_BATCHES:
            break
        imgs_dev = imgs.to(device)
        labels_dev = labels.to(device)

        spatial_map = model.features(imgs_dev)
        au_heatmaps = get_batch_au_heatmaps(imgs_dev, labels_dev)
        relevance = compute_channel_relevance_512(spatial_map, au_heatmaps)

        for i in range(imgs.size(0)):
            cls = labels[i].item()
            all_relevance[cls].append(relevance[i].cpu().numpy())

# Only visualize classes with data
active_classes = [c for c in range(7) if len(all_relevance[c]) > 0]
n_plots = len(active_classes)
cols = min(4, n_plots)
rows = (n_plots + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
if n_plots == 1:
    axes = np.array([axes])
axes = np.array(axes).flatten()

for plot_idx, cls_idx in enumerate(active_classes):
    ax = axes[plot_idx]
    avg_rel = np.mean(all_relevance[cls_idx], axis=0)
    uniform = 1.0 / 512

    ax.bar(range(512), sorted(avg_rel, reverse=True), width=1.0, alpha=0.7)
    ax.axhline(y=uniform, color='r', linestyle='--', label=f'Uniform={uniform:.4f}')
    ax.set_title(f'{class_names_display[cls_idx]} (n={len(all_relevance[cls_idx])})')
    ax.set_xlabel('Channel (sorted)')
    ax.set_ylabel('Relevance')
    ax.legend(fontsize=8)

    top_k = 51  # top 10%
    top_ratio = np.sum(sorted(avg_rel, reverse=True)[:top_k])
    ax.text(0.95, 0.95, f'Top10%: {top_ratio*100:.1f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

# Hide empty subplots
for idx in range(n_plots, len(axes)):
    axes[idx].axis('off')

fig.suptitle('Channel Relevance Distribution per Expression (sorted)', fontsize=14)
plt.tight_layout()
plt.show()

print('-> If uniform (1/512=0.195%), AU weighting has no effect')
print('-> If top 10% channels hold >20% of total weight, differentiation is working')

In [ ]:
# --- 3. Top channel overlap analysis across expressions ---
# Check if each expression uses different important channels

top_k = 50  # top 50 channels
top_channels = {}

active_classes = [c for c in range(7) if len(all_relevance[c]) > 0]

for cls_idx in active_classes:
    avg_rel = np.mean(all_relevance[cls_idx], axis=0)
    top_channels[cls_idx] = set(np.argsort(avg_rel)[-top_k:])

# Inter-expression top channel overlap (Jaccard similarity)
n_active = len(active_classes)
overlap_matrix = np.zeros((n_active, n_active))
for i_idx, i in enumerate(active_classes):
    for j_idx, j in enumerate(active_classes):
        intersection = len(top_channels[i] & top_channels[j])
        union = len(top_channels[i] | top_channels[j])
        overlap_matrix[i_idx][j_idx] = intersection / union if union > 0 else 0

active_names = [class_names_display[c] for c in active_classes]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(overlap_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n_active))
ax.set_xticklabels(active_names, rotation=45)
ax.set_yticks(range(n_active))
ax.set_yticklabels(active_names)
ax.set_title(f'Top-{top_k} Channel Overlap (Jaccard Similarity)')
ax.set_xlabel('Expression')
ax.set_ylabel('Expression')

for i in range(n_active):
    for j in range(n_active):
        ax.text(j, i, f'{overlap_matrix[i][j]:.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if overlap_matrix[i][j] > 0.5 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

off_diag = overlap_matrix.sum() - n_active
off_diag_count = n_active * n_active - n_active
avg_jaccard = off_diag / off_diag_count if off_diag_count > 0 else 0

print('-> Jaccard close to 1.0: all expressions use same channels (no differentiation)')
print('-> Jaccard low: each expression uses different channels (AU weighting effective)')
print(f'\nAverage Jaccard (off-diagonal): {avg_jaccard:.3f}')

In [ ]:
# --- 4. Sample images: Original + AU heatmap + ResNet activation comparison ---

def safe_denormalize(img_tensor):
    """denormalize_to_uint8 fallback"""
    try:
        return denormalize_to_uint8(img_tensor)
    except Exception:
        _mean = np.array([0.485, 0.456, 0.406])
        _std  = np.array([0.229, 0.224, 0.225])
        img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np * _std + _mean) * 255
        return np.clip(img_np, 0, 255).astype(np.uint8)

available_for_plot = sorted(samples_by_class.keys())
n_cols = len(available_for_plot)

fig, axes = plt.subplots(3, n_cols, figsize=(3 * n_cols, 9))
if n_cols == 1:
    axes = axes.reshape(3, 1)
row_labels = ['Original Image', 'AU Heatmap (target)', 'ResNet Activation (learned)']

cos_sims = []
with torch.no_grad():
    for plot_idx, cls_idx in enumerate(available_for_plot):
        img, label = samples_by_class[cls_idx]
        img_dev = img.to(device)
        label_dev = label.to(device)

        # Original image
        img_show = safe_denormalize(img[0])
        axes[0][plot_idx].imshow(img_show)
        axes[0][plot_idx].set_title(class_names_display[cls_idx], fontsize=12)
        axes[0][plot_idx].axis('off')

        # AU heatmap
        au_hm = get_batch_au_heatmaps(img_dev, label_dev)
        axes[1][plot_idx].imshow(au_hm[0].cpu().numpy(), cmap='hot', interpolation='bilinear')
        axes[1][plot_idx].axis('off')

        # ResNet spatial map (channel mean)
        spatial_map = model.features(img_dev)
        activation = F.relu(spatial_map[0].mean(dim=0)).cpu().numpy()
        activation = activation / max(activation.max(), 1e-6)
        axes[2][plot_idx].imshow(activation, cmap='hot', interpolation='bilinear')
        axes[2][plot_idx].axis('off')

        # Cosine similarity
        act_flat = torch.tensor(activation).flatten()
        au_flat = au_hm[0].cpu().flatten()
        cos = F.cosine_similarity(act_flat.unsqueeze(0), au_flat.unsqueeze(0)).item()
        cos_sims.append((class_names_display[cls_idx], cos))
        axes[2][plot_idx].set_xlabel(f'cos={cos:.3f}', fontsize=10)

for i, label in enumerate(row_labels):
    axes[i][0].set_ylabel(label, fontsize=11, rotation=90, labelpad=40, va='center')

fig.suptitle('AU Heatmap (target) vs ResNet Activation (learned)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Per-expression cosine similarity (AU heatmap vs ResNet activation):')
for name, cos in cos_sims:
    print(f'  {name}: {cos:.3f}')
avg_cos = sum(c for _, c in cos_sims) / len(cos_sims)
print(f'  Average: {avg_cos:.3f}')
print('-> Higher = ResNet activation aligns better with AU regions')

In [ ]:
# --- 5. AU Alignment: per-expression cosine similarity analysis ---
# Measure ResNet activation vs AU heatmap cosine similarity across entire test set

model.eval()
au_cos_by_class = {i: [] for i in range(7)}

with torch.no_grad():
    for imgs, labels, idxs, _ in test_loader:
        imgs_dev = imgs.to(device)
        labels_dev = labels.to(device)

        spatial_map = model.features(imgs_dev)
        au_heatmaps = get_batch_au_heatmaps(imgs_dev, labels_dev)

        act_map = F.relu(spatial_map.mean(dim=1))                    # (N, 7, 7)
        act_flat = act_map.view(act_map.size(0), -1)                 # (N, 49)
        act_norm = act_flat / act_flat.max(dim=1, keepdim=True)[0].clamp(min=1e-6)

        au_flat = au_heatmaps.view(au_heatmaps.size(0), -1)         # (N, 49)
        cos_sim = F.cosine_similarity(act_norm, au_flat, dim=1)      # (N,)

        for i in range(imgs.size(0)):
            cls = labels[i].item()
            au_cos_by_class[cls].append(cos_sim[i].item())

# Bar chart
active_classes = [c for c in range(7) if len(au_cos_by_class[c]) > 0]

fig, ax = plt.subplots(figsize=(10, 5))
names = []
means = []
for cls_idx in active_classes:
    vals = au_cos_by_class[cls_idx]
    names.append(class_names_display[cls_idx])
    means.append(np.mean(vals))

bars = ax.bar(names, means, color='steelblue', alpha=0.8)
ax.axhline(y=np.mean(means), color='r', linestyle='--', label=f'Average={np.mean(means):.3f}')
ax.set_ylabel('Cosine Similarity (AU heatmap vs ResNet activation)')
ax.set_title('AU Alignment: How well ResNet follows AU guidance')
ax.legend()
ax.set_ylim(0, 1)

for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('-> Close to 1.0: ResNet follows AU spatial guidance well')
print('-> Close to 0: ResNet ignores AU spatial guidance')

In [ ]:
# --- 6. Per-expression AU Alignment detail table ---

print('Expression-wise AU alignment (cosine similarity):')
print(f'{"Expression":<12} {"Mean cos":<10} {"Std":<10} {"n":<6}')
print('-' * 40)
for cls_idx in active_classes:
    vals = au_cos_by_class[cls_idx]
    print(f'{class_names_display[cls_idx]:<12} {np.mean(vals):<10.3f} {np.std(vals):<10.3f} {len(vals):<6}')

print()
print('-> Neutral has no AUs so uniform heatmap -> cosine sim may appear high (not meaningful)')
print('-> Higher values for other expressions indicate effective AU alignment')

## 12. Sanity Check
Run before training to verify model forward pass works correctly.

In [ ]:
print('Running sanity check...')
model.eval()

dummy_imgs   = torch.randn(4, 3, 224, 224).to(device)
dummy_labels = torch.randint(0, 7, (4,)).to(device)

with torch.no_grad():
    out, _ = model(dummy_imgs, clip_model, dummy_labels, phase='test')
print(f'Output shape (test) : {out.shape}  expected: [4, 7]')

model.train()
out, spatial_map, masked_features = model(dummy_imgs, clip_model, dummy_labels, phase='train')
dummy_au_heatmaps = torch.ones(4, 7, 7).to(device)
relevance_weights = compute_channel_relevance_512(spatial_map, dummy_au_heatmaps)
mc_loss  = supervisor_au_weighted(masked_features, dummy_labels, relevance_weights, cnum=73)
loss_cls = nn.CrossEntropyLoss()(out, dummy_labels)
total    = loss_cls + 1.5 * mc_loss[0] + 5.0 * mc_loss[1]

print(f'Output shape (train): {out.shape}  expected: [4, 7]')
print(f'spatial_map         : {spatial_map.shape}  expected: [4, 512, 7, 7]')
print(f'relevance_weights   : {relevance_weights.shape}  expected: [4, 512]')
print(f'l_cls : {loss_cls.item():.4f}')
print(f'l_sep : {mc_loss[0].item():.4f}  (AU-weighted)')
print(f'l_div : {mc_loss[1].item():.4f}  (unchanged)')
print(f'total : {total.item():.4f}')

model.eval()
print('Sanity check passed.')